# Lab: from text to BERT



A sentence such as *"The cat sat on the mat."* goes through three stages before BERT can predict a hidden word in it: a **tokenizer** splits it into subword units, **self-attention** mixes information between those units while ignoring padding, and a **masked-language-model (MLM)** head scores every vocabulary entry for each hidden position. In this lab you build or test each stage yourself, then use a pretrained BERT to see the stages working together.

**Learning objectives**

- Apply learned BPE merges and WordPiece longest-match rules to new words, and explain how each handles unknown characters.
- Measure how vocabulary size changes sequence length, and how one tokenizer can be much less efficient for some languages than for others.
- Implement scaled dot-product attention with a padding mask and prove, with a test, that padded positions cannot change the result.
- Read BERT's three input tensors and implement BERT's 80/10/10 masking rule.
- Query a pretrained BERT, measure how well it recovers masked tokens, and reason carefully about what a template-based bias probe can and cannot show.

**Prerequisites:** [Attention Is All You Need](../../transformers/Attention_Is_All_You_Need.ipynb), the [BPE](../../tokenizers/BPE_tokenizer.ipynb) and [WordPiece](../../tokenizers/wordpiece_tokenizer.ipynb) notebooks, and [BERT](../../transformers/BERT.ipynb).

**Marks (30)**

| Part | Exercise | Marks |
| --- | --- | --- |
| A. Tokenization | 1 BPE tokenizer · 2 WordPiece tokenizer · 3 vocabulary-size study · 4 tokenizers across languages | 2 + 2 + 2 + 2 |
| B. Attention | 5 masked attention · 6 padding mask · 7 leakage test | 3 + 1 + 2 |
| C. BERT | 8 input tensors · 9 MLM masking · 10 top-k predictions · 11 bias probe | 2 + 4 + 3 + 2 |
| All | Written answers: correct, specific, and supported by your outputs | 5 |

## Setup

In [ ]:
# Optional, only if a package is missing in your environment (Kaggle images already include these):
# !pip install -q transformers tokenizers datasets

import collections
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import datasets
import tokenizers
import transformers

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__, "| datasets", datasets.__version__)
print("device:", device, "(" + torch.cuda.get_device_name(0) + ")" if device.type == "cuda" else "")
if device.type != "cuda":
    print("No GPU found: Parts A and B are fine, Part C will be slow. On Kaggle, enable the GPU accelerator.")

PyTorch 2.1.2+cu121 | transformers 4.46.3 | tokenizers 0.20.3 | datasets 3.2.0
device: cuda (Tesla T4)


# Part A — Subword tokenization

Both BPE and WordPiece **learn** a vocabulary from a corpus and then **apply** it to new text. Training was covered in the notebooks; here you write the *application* step, which is what runs every time a model reads text.

| | BPE (GPT-2, RoBERTa) | WordPiece (BERT) |
| --- | --- | --- |
| What training produces | an ordered list of **merge rules** | a **vocabulary** of pieces (continuations start with `##`) |
| How a new word is split | start from characters, apply the merges **in the order they were learned** | **greedy longest match**: take the longest piece in the vocabulary that starts here, repeat |
| Unknown character | becomes `[UNK]`; the rest of the word is still split | the **whole word** becomes `[UNK]` |

## Exercise 1 — Apply BPE merges (2 marks)

Write `bpe_tokenize_word(word, merges, base_vocab)`.

- Start from the list of characters of `word`, replacing any character not in `base_vocab` with `"[UNK]"`.
- Go through `merges` **in order**. Each entry is `((a, b), merged)`. Replace every adjacent pair `a, b` in the current list by `merged`, scanning left to right, and move on to the next rule.

The toy rules below are the first three merges learned on the corpus `hug ×10, pug ×5, pun ×12, bun ×4, hugs ×5` from the BPE notebook.

In [ ]:
TOY_BASE_VOCAB = set("bghnpsu")
TOY_MERGES = [(("u", "g"), "ug"), (("u", "n"), "un"), (("h", "ug"), "hug")]

In [ ]:
def bpe_tokenize_word(word, merges, base_vocab, unk_token="[UNK]"):
    """Split one word by applying BPE merges in their learned order. Returns a list of tokens."""
    tokens = [c if c in base_vocab else unk_token for c in word]
    for (a, b), merged in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

In [ ]:
cases = {
    "hugs":  ["hug", "s"],
    "bug":   ["b", "ug"],
    "pun":   ["p", "un"],
    "mug":   ["[UNK]", "ug"],
    "thug":  ["[UNK]", "hug"],
    "unhug": ["un", "hug"],
}
for word, expected in cases.items():
    got = bpe_tokenize_word(word, TOY_MERGES, TOY_BASE_VOCAB)
    assert got == expected, f"{word!r}: expected {expected}, got {got}"
    print(f"{word:6s} -> {got}")
print("Exercise 1 checks passed.")

hugs   -> ['hug', 's']
bug    -> ['b', 'ug']
pun    -> ['p', 'un']
mug    -> ['[UNK]', 'ug']
thug   -> ['[UNK]', 'hug']
unhug  -> ['un', 'hug']
Exercise 1 checks passed.


**Question 1.** Suppose the merge list also contained `(("h", "u"), "hu")`, placed **before** `(("u", "g"), "ug")`. How would `hug` be tokenized now? What does this tell you about why merges must be applied in their learned order rather than, say, longest-first?

**Your answer — Question 1:**

If `(h, u) -> hu` came before `(u, g) -> ug`, then `hug` would first become `['hu', 'g']` after the first merge, and neither of the later merges would apply (there is no pair `(hu, g)` in the list). So the result would be `['hu', 'g']` instead of the correct `['hug']`.

This shows that merge order matters a lot. The BPE training algorithm learns merges in a specific frequency-driven order, and applying them out of order breaks the intended grouping. A longest-first approach would try to greedily match the longest token, which could merge characters that were never actually learned as a unit together, producing different (and usually worse) segmentations. The ordering encodes the hierarchy of how subwords were built up during training — common pairs first, rarer combinations later.

## Exercise 2 — WordPiece longest match (2 marks)

Write `wordpiece_tokenize_word(word, vocab)`.

- Start at position `start = 0`. Try the **longest** substring `word[start:end]` first and shorten it one character at a time until it is in `vocab`. Pieces that do not begin the word must be looked up **with** the `##` prefix.
- Append the match and continue from where it ended.
- If no piece matches at some position, return `["[UNK]"]` for the **whole word**.

The toy vocabulary below is hand-made so that the expected answers are easy to verify by hand.

In [ ]:
TOY_WP_VOCAB = {"[UNK]", "b", "h", "p", "u", "##u", "##g", "##n", "##s", "hu", "hug", "##gs", "##un"}

In [ ]:
def wordpiece_tokenize_word(word, vocab, unk_token="[UNK]", prefix="##"):
    """Split one word by greedy longest match. Returns a list of tokens, or [unk_token] for the whole word."""
    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        found = None
        while start < end:
            substr = word[start:end]
            if start > 0:
                substr = prefix + substr
            if substr in vocab:
                found = substr
                break
            end -= 1
        if found is None:
            return [unk_token]
        tokens.append(found)
        start = end
    return tokens

In [ ]:
cases = {
    "hugs":  ["hug", "##s"],          # longest match takes "hug", not "hu" + "##gs"
    "bun":   ["b", "##un"],
    "pugs":  ["p", "##u", "##gs"],
    "hu":    ["hu"],
    "mug":   ["[UNK]"],               # no piece starts with "m": the whole word is unknown
    "hugsx": ["[UNK]"],               # "x" fails at the end, and the whole word still becomes [UNK]
}
for word, expected in cases.items():
    got = wordpiece_tokenize_word(word, TOY_WP_VOCAB)
    assert got == expected, f"{word!r}: expected {expected}, got {got}"
    print(f"{word:6s} -> {got}")
print("Exercise 2 checks passed.")

hugs   -> ['hug', '##s']
bun    -> ['b', '##un']
pugs   -> ['p', '##u', '##gs']
hu     -> ['hu']
mug    -> ['[UNK]']
hugsx  -> ['[UNK]']
Exercise 2 checks passed.


**Question 2.** BPE turned `mug` into `['[UNK]', 'ug']`, while WordPiece turned it into `['[UNK]']`. Give one advantage of each behavior. Why does GPT-2's *byte-level* BPE never need an `[UNK]` token at all?

**Your answer — Question 2:**

BPE's behavior of keeping `['[UNK]', 'ug']` has the advantage that the model still gets partial information — it knows the word ends in "ug" even though it does not recognize the first character. This means the model can still pick up on the suffix pattern and potentially understand words like "rug" or "jug" even if it has never seen the leading character.

WordPiece returning `['[UNK]']` for the entire word is arguably cleaner because it does not pretend to understand a word it cannot fully parse. If one character fails, the whole tokenization is suspect, so it is more honest to just flag the whole thing as unknown rather than presenting a half-broken segmentation.

GPT-2's byte-level BPE never needs `[UNK]` because it operates on individual bytes rather than characters. Every possible byte value (0-255) is in the base vocabulary, so any character — including emoji, rare Unicode scripts, or even binary junk — can be represented as a sequence of bytes. There is literally no input byte that falls outside the vocabulary, so the unknown-token problem disappears entirely.

## Exercise 3 — How vocabulary size changes sequence length (2 marks)

Real tokenizers are trained on large corpora, so we now use the Hugging Face `tokenizers` library, a fast (Rust) implementation, on **WikiText-2**. The cell below downloads it and removes empty lines and section headings.

Each trained tokenizer first **pre-tokenizes** text into words (splitting on whitespace and punctuation), then splits each word into subwords. Two numbers summarize how efficient a tokenizer is on some text:

- **Fertility** = subword tokens per pre-tokenized word. A value of 1.0 means no word was split; larger values mean longer sequences.
- **Split rate** = fraction of words split into more than one token.

In [ ]:
from datasets import load_dataset

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def clean(lines):
    """Drop empty lines and '= Heading =' lines."""
    return [s.strip() for s in lines if s.strip() and not s.strip().startswith("=")]

train_texts, val_texts = clean(wiki["train"]["text"]), clean(wiki["validation"]["text"])
print(f"train paragraphs: {len(train_texts):,} | validation paragraphs: {len(val_texts):,}")
print("example:", val_texts[0][:200], "...")

train paragraphs: 14,524 | validation paragraphs: 1,527
example: Robert <unk> is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by his role as combatant in the  ...


In [ ]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

def train_subword_tokenizer(kind, texts, vocab_size):
    """Train a lowercasing BPE or WordPiece tokenizer on `texts` (training split only)."""
    if kind == "bpe":
        tok = Tokenizer(models.BPE(unk_token="[UNK]"))
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    elif kind == "wordpiece":
        tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
        trainer = trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
    else:
        raise ValueError(kind)
    tok.normalizer = normalizers.Lowercase()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()      # words = runs of letters/digits, or of punctuation
    tok.train_from_iterator(texts, trainer=trainer)
    return tok

demo = train_subword_tokenizer("wordpiece", train_texts, vocab_size=4000)
enc = demo.encode("Tokenization is unbelievably useful.")
print("tokens  :", enc.tokens)
print("word_ids:", enc.word_ids, " <- index of the pre-tokenized word each token came from")

tokens  : ['token', '##ization', 'is', 'un', '##bel', '##ie', '##va', '##bly', 'useful', '.']
word_ids: [0, 0, 1, 2, 2, 2, 2, 2, 3, 4]  <- index of the pre-tokenized word each token came from


Write `subword_stats(tokenizer, texts)`. Use `tokenizer.encode_batch(texts)`, and for each encoding use `enc.word_ids` (as printed above) to count how many tokens each word produced. Return a dictionary with `fertility`, `split_rate`, and `unk_rate` (fraction of all tokens equal to `"[UNK]"`).

In [ ]:
def subword_stats(tokenizer, texts):
    """Return {"fertility": tokens per word, "split_rate": fraction of words with >1 token, "unk_rate": [UNK] share}."""
    n_words = n_tokens = n_split = n_unk = 0
    for enc in tokenizer.encode_batch(texts):
        word_token_counts = {}
        for wid in enc.word_ids:
            if wid is not None:
                word_token_counts[wid] = word_token_counts.get(wid, 0) + 1
        n_words += len(word_token_counts)
        n_tokens += sum(word_token_counts.values())
        n_split += sum(1 for c in word_token_counts.values() if c > 1)
        n_unk += sum(1 for t in enc.tokens if t == "[UNK]")
    return {"fertility": n_tokens / n_words, "split_rate": n_split / n_words, "unk_rate": n_unk / n_tokens}

In [ ]:
toy = subword_stats(demo, ["tokenization is useful"])
assert set(toy) == {"fertility", "split_rate", "unk_rate"}
assert toy["fertility"] >= 1.0 and 0.0 <= toy["split_rate"] <= 1.0
manual = demo.encode("tokenization is useful")
assert abs(toy["fertility"] - len(manual.tokens) / 3) < 1e-9, "fertility should be tokens / words (3 words here)"
print("Exercise 3 checks passed. Now running the vocabulary-size study (a minute or so)...")

VOCAB_SIZES = [1000, 4000, 16000]
study = {}
for kind in ("bpe", "wordpiece"):
    for size in VOCAB_SIZES:
        tok = train_subword_tokenizer(kind, train_texts, size)              # trained on train only
        study[(kind, size)] = (tok, subword_stats(tok, val_texts))          # measured on validation
        s = study[(kind, size)][1]
        print(f"{kind:9s} V={size:6,d}  fertility={s['fertility']:.3f}  split_rate={s['split_rate']:.3f}  unk_rate={s['unk_rate']:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
for kind, marker in (("bpe", "o"), ("wordpiece", "s")):
    ax.plot(VOCAB_SIZES, [study[(kind, v)][1]["fertility"] for v in VOCAB_SIZES], marker=marker, label=kind)
ax.set_xscale("log"); ax.set_xticks(VOCAB_SIZES); ax.set_xticklabels([f"{v:,}" for v in VOCAB_SIZES])
ax.set_xlabel("vocabulary size (log scale)"); ax.set_ylabel("tokens per word (validation)")
ax.set_title("Larger vocabularies give shorter sequences"); ax.legend(); plt.tight_layout(); plt.show()

for word in ["photosynthesis", "unbelievably", "bennett", "transformers", "kaggle"]:
    print(f"{word:15s}", " | ".join(f"{k[0]} {v:>5}: {study[(k, v)][0].encode(word).tokens}"
                                    for k in ("bpe", "wordpiece") for v in (1000, 16000)))

Exercise 3 checks passed. Now running the vocabulary-size study (a minute or so)...
bpe       V= 1,000  fertility=1.536  split_rate=0.356  unk_rate=0.0000
bpe       V= 4,000  fertility=1.230  split_rate=0.174  unk_rate=0.0000
bpe       V=16,000  fertility=1.083  split_rate=0.071  unk_rate=0.0000
wordpiece V= 1,000  fertility=1.576  split_rate=0.371  unk_rate=0.0032
wordpiece V= 4,000  fertility=1.238  split_rate=0.178  unk_rate=0.0004
wordpiece V=16,000  fertility=1.088  split_rate=0.074  unk_rate=0.0001
photosynthesis  bpe  1000: ['ph', 'ot', 'os', 'yn', 'the', 'sis'] | bpe 16000: ['photos', 'ynthesis'] | wordpiece  1000: ['ph', '##ot', '##os', '##yn', '##th', '##es', '##is'] | wordpiece 16000: ['photo', '##synthesis']
unbelievably    bpe  1000: ['un', 'be', 'li', 'ev', 'ab', 'ly'] | bpe 16000: ['unbeliev', 'ably'] | wordpiece  1000: ['un', '##be', '##li', '##ev', '##ab', '##ly'] | wordpiece 16000: ['un', '##believably']
bennett         bpe  1000: ['ben', 'ne', 'tt'] | bpe 16000: ['be

**Question 3.** (a) Describe the trend in fertility as the vocabulary grows, using your numbers. (b) The vocabulary also sets the size of the embedding table. For a model with hidden size $d = 768$, how many embedding parameters does a 16,000-token vocabulary need, compared with 1,000 tokens? (c) Using (a) and (b), explain the trade-off a model designer faces when choosing the vocabulary size. Why are all the measurements taken on the *validation* paragraphs?

**Your answer — Question 3:**

(a) Fertility drops consistently as the vocabulary gets bigger. For BPE it went from 1.536 at V=1000 down to 1.230 at V=4000 and 1.083 at V=16000. WordPiece shows the same pattern: 1.576, 1.238, 1.088. Basically, with more subword units available, fewer words need to be split, so the sequences get shorter. The split rate tells the same story — only about 7% of words get fragmented at V=16000 versus 35-37% at V=1000.

(b) The embedding table has V x d parameters. At d=768: a 16,000-token vocabulary needs 16,000 x 768 = 12,288,000 parameters, while 1,000 tokens only needs 768,000. That is a 16x increase — over 11.5 million extra parameters just for the embeddings.

(c) The trade-off is between sequence length and model size. A bigger vocabulary makes sequences shorter, which is great because self-attention scales quadratically with sequence length, and shorter sequences also mean more text fits in the context window. But a bigger vocabulary means a larger embedding table and a larger output projection layer, which adds parameters and memory. The sweet spot depends on the domain: if you are dealing with very long documents, shorter sequences might be worth the extra parameters. Most practical models settle on something like 30k-50k tokens as a compromise.

We measure on the validation paragraphs rather than the training ones because the tokenizer was trained on the training data. If we measured on training data, the stats would be optimistic — the tokenizer would look better than it actually performs on unseen text, since it was specifically built to handle the training vocabulary well.

## Exercise 4 — One sentence, three pretrained tokenizers, three languages (2 marks)

A tokenizer trained mostly on one language can be very inefficient on others. We compare three real tokenizers on the same short sentence written in English, Hindi (Devanagari script), and Hinglish (Hindi written in the Latin alphabet, as in many chats).

In [ ]:
from transformers import AutoTokenizer

PRETRAINED = ["google-bert/bert-base-uncased", "google-bert/bert-base-multilingual-cased", "openai-community/gpt2"]
pretrained_toks = {name: AutoTokenizer.from_pretrained(name) for name in PRETRAINED}   # tokenizers only: small downloads

SENTENCES = {
    "English":  "I like Indian food very much.",
    "Hindi":    "मुझे भारतीय खाना बहुत पसंद है।",
    "Hinglish": "Mujhe Indian khana bahut pasand hai.",
}

In [ ]:
def tokens_per_word(tokenizer, text):
    """Tokens produced by tokenizer.tokenize(text), divided by the number of whitespace-separated words."""
    return len(tokenizer.tokenize(text)) / len(text.split())

In [ ]:
assert abs(tokens_per_word(pretrained_toks[PRETRAINED[0]], "hello world") - len(pretrained_toks[PRETRAINED[0]].tokenize("hello world")) / 2) < 1e-9
print("Exercise 4 checks passed.\n")

print(f"{'':10s}" + "".join(f"{name.split('/')[-1]:>30s}" for name in PRETRAINED))
for lang, text in SENTENCES.items():
    print(f"{lang:10s}" + "".join(f"{tokens_per_word(pretrained_toks[n], text):30.2f}" for n in PRETRAINED))

for lang, text in SENTENCES.items():
    print(f"\n{lang}: {text}")
    for name in PRETRAINED:
        tok = pretrained_toks[name]
        pieces = tok.tokenize(text)
        print(f"  {name.split('/')[-1]:28s} {len(pieces):2d} tokens: {pieces}")
        if "uncased" in name:
            print(f"  {'':28s} decoded back: {tok.decode(tok.encode(text, add_special_tokens=False))!r}")

Exercise 4 checks passed.

                    bert-base-uncased  bert-base-multilingual-cased                       gpt2
English            1.00                          1.00                          1.00
Hindi              4.14                          1.43                          8.43
Hinglish           1.33                          1.17                          1.17

English: I like Indian food very much.
  bert-base-uncased              7 tokens: ['i', 'like', 'indian', 'food', 'very', 'much', '.']
                             decoded back: 'i like indian food very much .'
  bert-base-multilingual-cased   7 tokens: ['I', 'like', 'Indian', 'food', 'very', 'much', '.']
  gpt2                           7 tokens: ['I', 'Ġlike', 'ĠIndian', 'Ġfood', 'Ġvery', 'Ġmuch', '.']

Hindi: मुझे भारतीय खाना बहुत पसंद है।
  bert-base-uncased             29 tokens: ['म', '##ु', '##झ', '##े', 'भ', '##ा', '##र', '##त', '##ी', '##य', 'ख', '##ा', '##ना', 'ब', '##ह', '##ु', '##त', 'प', '##स', '##ं', '##

**Question 4.** (a) Which tokenizer is most efficient for each language, and which is least efficient for Hindi? Explain using the table above. (b) Compare the decoded `bert-base-uncased` output for Hindi with the original sentence. What has happened? (c) Context windows are measured in tokens, and paid APIs often charge per token. What does your table imply for users who write in Hindi?

**Your answer — Question 4:**

(a) For English, all three tokenizers are equally efficient at 1.0 tokens per word — makes sense since they were all trained on lots of English. For Hindi, the multilingual BERT is by far the best at 1.43 tokens/word because its vocabulary was specifically designed to cover ~100 languages including Hindi. The worst for Hindi is GPT-2 at 8.43 tokens/word — it shreds each Devanagari character into multiple byte-level tokens because its vocabulary was built almost entirely from English web text. For Hinglish, multilingual BERT and GPT-2 tie at about 1.17, with bert-base-uncased slightly worse at 1.33.

(b) Looking at the decoded output from bert-base-uncased on Hindi, it says 'मुझे भारतीय खाना बहुत पसंद हैं [UNK]' — the sentence-ending punctuation mark (।, the Devanagari danda) got turned into [UNK] because that character was not in the vocabulary at all. The rest of the text survived but was broken into individual character-level tokens, which is extremely wasteful. The tokenizer technically handled it but in the most inefficient way possible.

(c) The implications for Hindi users are pretty rough. A sentence that takes 7 tokens in English takes 59 tokens with GPT-2 for Hindi — roughly 8x more. This means Hindi users can fit about 8x less text in a given context window, and if they are paying per token, they pay about 8x more for the same amount of content. This is a real equity issue with current LLMs. Using a multilingual tokenizer helps a lot (only 10 tokens for the same Hindi sentence), but even then Hindi is less efficiently encoded than English. This kind of disparity affects everyone who communicates in non-Latin-script languages.

# Part B — Self-attention with a padding mask

Sentences in a batch have different lengths, so shorter ones are **padded** to a common length. Self-attention lets every position read every other position, so without a mask the real tokens would also read the padding, and their representations would depend on how much padding the batch happened to need.

We use the same convention as [Attention Is All You Need](../../transformers/Attention_Is_All_You_Need.ipynb): a Boolean mask in which **`True` means "may attend"** and `False` means "blocked".

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V,\qquad M_{ij}=\begin{cases}0&\text{key } j \text{ allowed}\\\\-\infty&\text{key } j \text{ blocked}\end{cases}$$

## Exercise 5 — Scaled dot-product attention with a mask (3 marks)

Write `scaled_dot_product_attention(q, k, v, mask=None)`.

- `q`: `(B, H, T_q, d_k)`, `k` and `v`: `(B, H, T_k, d_k)`. `mask`: Boolean, broadcastable to `(B, H, T_q, T_k)`, `True` = may attend.
- Compute the scores, scale by $\sqrt{d_k}$, set blocked scores to `-inf` (use `masked_fill`), apply softmax over the **key** dimension, and combine the values.
- Return `(output, weights)` with shapes `(B, H, T_q, d_k)` and `(B, H, T_q, T_k)`.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """Returns (output, weights). mask: bool, broadcastable to (B, H, T_q, T_k); True = may attend."""
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, v)
    return output, weights

In [ ]:
g = torch.Generator().manual_seed(0)
B, H, T, d_k = 2, 3, 5, 4
q, k, v = (torch.randn(B, H, T, d_k, generator=g) for _ in range(3))
lengths = torch.tensor([5, 3])                                            # sequence 2 has 2 padded positions
key_mask = (torch.arange(T)[None, :] < lengths[:, None])[:, None, None, :]   # (B, 1, 1, T), True = real token

out, w = scaled_dot_product_attention(q, k, v, key_mask)
assert out.shape == (B, H, T, d_k) and w.shape == (B, H, T, T), "wrong output shapes"
assert torch.allclose(w.sum(-1), torch.ones(B, H, T)), "each row of weights must sum to 1"
assert (w[1, :, :, 3:] == 0).all(), "padded keys must receive exactly zero weight"
ref = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
assert torch.allclose(out, ref, atol=1e-5), "does not match torch's scaled_dot_product_attention"
out_nomask, _ = scaled_dot_product_attention(q, k, v)
assert torch.allclose(out_nomask, F.scaled_dot_product_attention(q, k, v), atol=1e-5), "mask=None case is wrong"
print("Exercise 5 checks passed.")

Exercise 5 checks passed.


## Exercise 6 — The padding mask (1 mark)

Write `make_padding_mask(input_ids, pad_id)`: given token IDs of shape `(B, T)`, return a Boolean mask of shape `(B, 1, 1, T)` that is `True` at real tokens.

In [ ]:
def make_padding_mask(input_ids, pad_id):
    """(B, T) token IDs -> (B, 1, 1, T) bool mask, True at non-padding positions."""
    return (input_ids != pad_id).unsqueeze(1).unsqueeze(1)

In [ ]:
ids = torch.tensor([[5, 6, 7, 0, 0], [8, 9, 10, 11, 12]])
m = make_padding_mask(ids, pad_id=0)
assert m.shape == (2, 1, 1, 5) and m.dtype == torch.bool
assert m[0, 0, 0].tolist() == [True, True, True, False, False] and m[1].all()
print("Exercise 6 checks passed.")

Exercise 6 checks passed.


## Provided: a small Transformer encoder

Read this code; you do not need to change it.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q, self.W_k, self.W_v, self.W_o = (nn.Linear(d_model, d_model) for _ in range(4))

    def forward(self, x, mask=None):                                   # x: (B, T, D)
        B, T, D = x.shape
        split = lambda t: t.view(B, T, self.h, self.d_k).transpose(1, 2)        # (B, H, T, d_k)
        out, weights = scaled_dot_product_attention(split(self.W_q(x)), split(self.W_k(x)), split(self.W_v(x)), mask)
        return self.W_o(out.transpose(1, 2).reshape(B, T, D)), weights          # merge heads -> (B, T, D)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        a, _ = self.attn(x, mask)
        x = self.norm1(x + self.drop(a))                  # residual + layer norm (post-norm, as in the paper)
        return self.norm2(x + self.drop(self.ff(x)))


class TinyEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_heads=4, d_ff=64, n_layers=2, max_len=64, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
        self.layers = nn.ModuleList(EncoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers))

    def forward(self, ids, mask=None):                   # ids: (B, T)
        x = self.emb(ids) * math.sqrt(self.emb.embedding_dim) + self.pe[: ids.size(1)]
        for layer in self.layers:
            x = layer(x, mask)
        return x                                          # (B, T, D)

torch.manual_seed(SEED)
encoder = TinyEncoder(vocab_size=50).eval()               # eval(): dropout off, so outputs are deterministic
print(encoder(torch.tensor([[3, 4, 5, 0]]), make_padding_mask(torch.tensor([[3, 4, 5, 0]]), 0)).shape)

torch.Size([1, 4, 32])


## Exercise 7 — Prove that padding cannot leak (2 marks)

Write `padding_leakage(model, ids, junk_ids, pad_id, use_mask)`.

In [ ]:
@torch.no_grad()
def padding_leakage(model, ids, junk_ids, pad_id, use_mask=True):
    """Max |output difference| at real positions when only padded positions change."""
    mask = make_padding_mask(ids, pad_id) if use_mask else None
    out1 = model(ids, mask)
    out2 = model(junk_ids, mask)
    real = ids != pad_id
    diff = (out1 - out2).abs()
    return diff[real.unsqueeze(-1).expand_as(diff)].max().item()

In [ ]:
ids = torch.tensor([[7, 8, 9, 0, 0, 0], [10, 11, 12, 13, 14, 0]])
junk_ids = ids.clone()
junk_ids[ids == 0] = torch.tensor([31, 32, 33, 34])       # change only the padded positions
with_mask = padding_leakage(encoder, ids, junk_ids, pad_id=0, use_mask=True)
without_mask = padding_leakage(encoder, ids, junk_ids, pad_id=0, use_mask=False)
print(f"with mask   : max change at real positions = {with_mask:.2e}")
print(f"without mask: max change at real positions = {without_mask:.2e}")
assert with_mask < 1e-5, "with the mask, real positions must not depend on padded content"
assert without_mask > 1e-3, "without the mask, padded content should leak into real positions"
print("Exercise 7 checks passed.")

with mask   : max change at real positions = 0.00e+00
without mask: max change at real positions = 3.74e-01
Exercise 7 checks passed.


**Question 5 (shapes and masks).** (a) For the encoder above with `B = 2`, `T = 6`, `D = 32`, `H = 4`, give the shapes of: the input IDs, the embeddings, `q` after splitting into heads, the attention weights, and the encoder output. (b) The padding mask hides padded **keys**, but the padded positions are still **queries** and still produce output vectors. Why is that harmless for BERT? (c) BERT uses a padding mask but no causal mask, while GPT-2 uses both. Explain why in one or two sentences each.

**Your answer — Question 5:**

(a) Shapes:
- Input IDs: (2, 6)
- Embeddings (after adding positional encoding): (2, 6, 32)
- q after splitting into heads: (2, 4, 6, 8) — 4 heads, each with d_k = 32/4 = 8
- Attention weights: (2, 4, 6, 6) — each query attends over all 6 key positions
- Encoder output: (2, 6, 32)

(b) Padded positions still produce output vectors, but that is harmless because downstream tasks never use those positions. BERT reads the [CLS] token for classification or reads specific non-pad token positions for token-level tasks. The padded query outputs are just wasted computation that gets thrown away — they do not contaminate the real positions because the mask prevents information flowing from pad keys into real queries.

(c) BERT uses only a padding mask because it is a bidirectional model — every real token is supposed to see every other real token, and only pad tokens need to be hidden. GPT-2 needs both a padding mask (to ignore padding in batched inputs) and a causal mask (to prevent each token from seeing future tokens), because it is an autoregressive model that generates left to right — looking ahead would be cheating during both training and generation.

# Part C — BERT: inputs, objective, and predictions

We now load the pretrained `bert-base-uncased` with its masked-language-model head (about 440 MB on the first run) and move it to the GPU.

In [ ]:
from transformers import AutoModelForMaskedLM

BERT = "google-bert/bert-base-uncased"
bert_tok = AutoTokenizer.from_pretrained(BERT)
bert_mlm = AutoModelForMaskedLM.from_pretrained(BERT).to(device).eval()     # eval(): dropout off
print("vocabulary size:", bert_tok.vocab_size, "| parameters:", round(sum(p.numel() for p in bert_mlm.parameters()) / 1e6, 1), "M")
print("special tokens:", {t: bert_tok.convert_tokens_to_ids(t) for t in ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]})

vocabulary size: 30522 | parameters: 109.5 M
special tokens: {'[PAD]': 0, '[UNK]': 100, '[CLS]': 101, '[SEP]': 102, '[MASK]': 103}


## Exercise 8 — BERT's three input tensors (2 marks)

In [ ]:
pair = bert_tok("The cat sat on the mat.", "It looked very comfortable.",
                padding="max_length", max_length=20, return_tensors="pt")
print(f"{'pos':>3}  {'token':12s} {'input_id':>8} {'type':>4} {'attn':>4}")
for i, (tid, tt, am) in enumerate(zip(pair["input_ids"][0], pair["token_type_ids"][0], pair["attention_mask"][0])):
    print(f"{i:3d}  {bert_tok.convert_ids_to_tokens(int(tid)):12s} {int(tid):8d} {int(tt):4d} {int(am):4d}")

pos  token         input_id type attn
  0  [CLS]              101    0    1
  1  the               1996    0    1
  2  cat               4937    0    1
  3  sat               2938    0    1
  4  on                2006    0    1
  5  the               1996    0    1
  6  mat               13523    0    1
  7  .                 1012    0    1
  8  [SEP]              102    0    1
  9  it                2009    1    1
 10  looked            2246    1    1
 11  very              2200    1    1
 12  comfortable       6242    1    1
 13  .                 1012    1    1
 14  [SEP]              102    1    1
 15  [PAD]                0    0    0
 16  [PAD]                0    0    0
 17  [PAD]                0    0    0
 18  [PAD]                0    0    0
 19  [PAD]                0    0    0


In [ ]:
def describe_bert_input(enc, tokenizer):
    """Summarize one encoded sentence pair (batch size 1)."""
    ids = enc["input_ids"][0]
    types = enc["token_type_ids"][0]
    attn = enc["attention_mask"][0]
    cls_id = tokenizer.convert_tokens_to_ids("[CLS]")
    sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
    cls_position = (ids == cls_id).nonzero(as_tuple=True)[0][0].item()
    sep_positions = (ids == sep_id).nonzero(as_tuple=True)[0].tolist()
    segment_b_tokens = int(((types == 1) & (attn == 1)).sum().item())
    pad_tokens = int((attn == 0).sum().item())
    return {"cls_position": cls_position, "sep_positions": sep_positions,
            "segment_b_tokens": segment_b_tokens, "pad_tokens": pad_tokens}

In [ ]:
info = describe_bert_input(pair, bert_tok)
print(info)
tokens_b = bert_tok.tokenize("It looked very comfortable.")
assert info["cls_position"] == 0
assert len(info["sep_positions"]) == 2
assert info["segment_b_tokens"] == len(tokens_b) + 1, "segment B = sentence-B tokens plus its closing [SEP]"
assert info["pad_tokens"] == 20 - (info["sep_positions"][-1] + 1)
print("Exercise 8 checks passed.")

{'cls_position': 0, 'sep_positions': [8, 14], 'segment_b_tokens': 6, 'pad_tokens': 5}
Exercise 8 checks passed.


**Question 6.** Which segment does the *first* `[SEP]` belong to, and which the second? What `token_type_ids` do padding positions get, and why does it not matter? What would `[CLS]`'s final hidden state be used for when BERT is later adapted to a sentence-pair classification task?

**Your answer — Question 6:**

The first [SEP] (at position 8) belongs to segment A — it has token_type_id 0, which closes off the first sentence. The second [SEP] (at position 14) belongs to segment B — token_type_id 1.

Padding positions get token_type_ids = 0, but this does not matter because the attention mask is 0 at those positions, so they are completely ignored during attention computation. BERT never reads from or writes to those positions in any meaningful way, so what segment they are labeled with is irrelevant.

For a sentence-pair classification task (like natural language inference or paraphrase detection), [CLS]'s final hidden state is used as the aggregate representation of the entire input pair. A classification head (typically a simple linear layer) is placed on top of [CLS]'s hidden vector to predict the class label. The idea is that through self-attention across all 12 layers, [CLS] has had the chance to absorb information from both sentences and can serve as a fixed-length summary of the relationship between them.

## Exercise 9 — BERT's 80/10/10 masking rule (4 marks)

In [ ]:
# A batch of real text: 512 validation paragraphs, truncated to 128 tokens.
mlm_batch = bert_tok(val_texts[:512], padding=True, truncation=True, max_length=128,
                     return_tensors="pt", return_special_tokens_mask=True)
special_mask = mlm_batch["special_tokens_mask"].bool() | (mlm_batch["attention_mask"] == 0)   # never select these
print("batch shape:", tuple(mlm_batch["input_ids"].shape), "| eligible tokens:", int((~special_mask).sum()))

batch shape: (512, 128) | eligible tokens: 38472


In [ ]:
def mask_tokens(input_ids, special_mask, mask_token_id, vocab_size, mlm_prob=0.15, generator=None):
    """BERT MLM masking. Returns (masked_ids, labels); labels are -100 where no prediction is required."""
    labels = input_ids.clone()
    masked_ids = input_ids.clone()
    probability_matrix = torch.full(input_ids.shape, mlm_prob)
    probability_matrix[special_mask] = 0.0
    selected = torch.bernoulli(probability_matrix, generator=generator).bool()
    labels[~selected] = -100
    mask_indices = torch.bernoulli(torch.full(input_ids.shape, 0.8), generator=generator).bool() & selected
    masked_ids[mask_indices] = mask_token_id
    random_indices = torch.bernoulli(torch.full(input_ids.shape, 0.5), generator=generator).bool() & selected & ~mask_indices
    random_tokens = torch.randint(vocab_size, input_ids.shape, generator=generator)
    masked_ids[random_indices] = random_tokens[random_indices]
    return masked_ids, labels

In [ ]:
original = mlm_batch["input_ids"].clone()
masked_ids, labels = mask_tokens(mlm_batch["input_ids"], special_mask, bert_tok.mask_token_id, bert_tok.vocab_size,
                                 mlm_prob=0.15, generator=torch.Generator().manual_seed(SEED))

assert torch.equal(mlm_batch["input_ids"], original), "input_ids was modified in place"
selected = labels != -100
assert not (selected & special_mask).any(), "special tokens or padding were selected"
assert torch.equal(labels[selected], original[selected]), "labels must hold the ORIGINAL token at selected positions"
assert torch.equal(masked_ids[~selected], original[~selected]), "non-selected positions must be unchanged"

eligible = int((~special_mask).sum()); n_sel = int(selected.sum())
is_mask = (masked_ids == bert_tok.mask_token_id) & selected
is_unchanged = (masked_ids == original) & selected
is_random = selected & ~is_mask & ~is_unchanged
frac = lambda m: int(m.sum()) / n_sel
print(f"selected: {n_sel / eligible:.3f} of eligible tokens (target 0.15)")
print(f"of selected -> [MASK]: {frac(is_mask):.3f} (0.8) | random: {frac(is_random):.3f} (~0.1) | unchanged: {frac(is_unchanged):.3f} (~0.1)")
assert abs(n_sel / eligible - 0.15) < 0.01
assert abs(frac(is_mask) - 0.8) < 0.03 and abs(frac(is_random) - 0.1) < 0.03 and abs(frac(is_unchanged) - 0.1) < 0.03
print("Exercise 9 checks passed.")

selected: 0.150 of eligible tokens (target 0.15)
of selected -> [MASK]: 0.800 (0.8) | random: 0.098 (~0.1) | unchanged: 0.102 (~0.1)
Exercise 9 checks passed.


**Question 7.** (a) Why not simply replace all selected tokens by `[MASK]`? (b) Why keep 10% unchanged and replace 10% with random tokens? (c) A random replacement occasionally picks the original token, so the "random" share is slightly below 0.1. Why is that harmless?

**Your answer — Question 7:**

(a) If all selected tokens were always replaced by [MASK], the model would learn that it only ever needs to predict something when it sees the [MASK] token. But during fine-tuning and actual use, there is no [MASK] token — the model sees normal text. This creates a mismatch between pretraining and usage. By sometimes keeping the original token or using a random one, the model learns that it might need to predict the correct token at any position, not just where it sees [MASK].

(b) The 10% unchanged serves a specific purpose: it teaches the model that even when the input token looks correct, it should still maintain a good representation of what the token should be. The model cannot just "trust" the input, because sometimes the input is right and sometimes it is wrong. The 10% random tokens force the model to rely on context rather than just copying the input — if the input might be garbage, the model has to actually understand the surrounding context to figure out the real word.

(c) When a random replacement happens to pick the original token, that position effectively becomes an "unchanged" position. This slightly reduces the measured random fraction and increases the unchanged fraction, but it is completely harmless because it is statistically rare (probability 1/30522 for any given token) and the model's learning objective does not change — it still has to predict the original token regardless of what is in the input slot.

## Exercise 10 — BERT's top-k predictions (3 marks)

In [ ]:
@torch.no_grad()
def top_k_predictions(text, k=5):
    """[(token, probability), ...] for the single [MASK] in text, most probable first."""
    enc = bert_tok(text, return_tensors="pt").to(device)
    logits = bert_mlm(**enc).logits
    mask_idx = (enc["input_ids"][0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0][0]
    probs = torch.softmax(logits[0, mask_idx], dim=-1)
    topk = torch.topk(probs, k)
    return [(bert_tok.convert_ids_to_tokens(idx.item()), prob.item())
            for idx, prob in zip(topk.indices, topk.values)]

In [ ]:
from transformers import pipeline

preds = top_k_predictions("The cat sat on the [MASK].", k=5)
print(preds)
probs_ = [p for _, p in preds]
assert len(preds) == 5 and probs_ == sorted(probs_, reverse=True) and 0 < sum(probs_) <= 1 + 1e-6
fill = pipeline("fill-mask", model=bert_mlm, tokenizer=bert_tok, device=0 if device.type == "cuda" else -1)
ref = fill("The cat sat on the [MASK].", top_k=1)[0]
assert bert_tok.convert_tokens_to_ids(preds[0][0]) == ref["token"], "top-1 token differs from the fill-mask pipeline"
assert abs(preds[0][1] - ref["score"]) < 1e-3, "top-1 probability differs from the fill-mask pipeline"
print("Exercise 10 checks passed.\n")

for text in ["Paris is the capital of [MASK].",
             "She opened the [MASK] and walked into the room.",
             "The doctor gave me some [MASK] for my headache.",
             "Bennett University is located in [MASK]."]:
    print(f"{text:50s}", [(t, round(p, 3)) for t, p in top_k_predictions(text)])

[('floor', 0.209), ('ground', 0.119), ('couch', 0.082), ('bed', 0.071), ('mat', 0.062)]
Exercise 10 checks passed.

Paris is the capital of [MASK].                    [('france', 0.988), ('paris', 0.002), ('europe', 0.001), ('belgium', 0.001), ('italy', 0.001)]
She opened the [MASK] and walked into the room.    [('door', 0.895), ('window', 0.018), ('gate', 0.013), ('curtain', 0.008), ('lid', 0.005)]
The doctor gave me some [MASK] for my headache.    [('medicine', 0.249), ('medication', 0.137), ('pills', 0.092), ('advice', 0.087), ('drugs', 0.063)]
Bennett University is located in [MASK].           [('toronto', 0.074), ('london', 0.048), ('australia', 0.035), ('california', 0.030), ('canada', 0.026)]


### Provided: how often does BERT recover masked tokens?

In [ ]:
@torch.no_grad()
def mlm_eval(masked_ids, labels, attention_mask, batch_size=64):
    correct = {"[MASK]": 0, "random": 0, "unchanged": 0}; total = dict.fromkeys(correct, 0); losses = []
    for s in range(0, masked_ids.size(0), batch_size):
        ids_b, lab_b, att_b = (t[s:s + batch_size].to(device) for t in (masked_ids, labels, attention_mask))
        out = bert_mlm(input_ids=ids_b, attention_mask=att_b, labels=lab_b)    # HF computes CE, ignoring -100
        losses.append(out.loss.item())
        pred = out.logits.argmax(-1)
        sel = lab_b != -100
        kinds = {"[MASK]": sel & (ids_b == bert_tok.mask_token_id),
                 "unchanged": sel & (ids_b == lab_b)}
        kinds["random"] = sel & ~kinds["[MASK]"] & ~kinds["unchanged"]
        for name, m in kinds.items():
            correct[name] += int((pred[m] == lab_b[m]).sum()); total[name] += int(m.sum())
    for name in correct:
        print(f"{name:10s} positions: {total[name]:6d}   top-1 accuracy = {correct[name] / max(total[name], 1):.3f}")
    print(f"mean MLM loss over batches = {sum(losses) / len(losses):.3f} nats/token")

mlm_eval(masked_ids, labels, mlm_batch["attention_mask"])

[MASK]     positions:   4618   top-1 accuracy = 0.647
random     positions:    567   top-1 accuracy = 0.352
unchanged  positions:    588   top-1 accuracy = 0.853
mean MLM loss over batches = 1.783 nats/token


**Question 8.** Rank the three kinds of positions by accuracy, and explain the ranking. Why would it be wrong to report the accuracy at `unchanged` positions as evidence of how well BERT 'understands' text?

**Your answer — Question 8:**

The ranking from highest to lowest accuracy is: unchanged (0.853) > [MASK] (0.647) > random (0.352).

This ordering makes sense when you think about what information the model has at each kind of position:

- **Unchanged positions** (85.3%): the original token is sitting right there in the input. The model can literally just "copy" the input token to the output. It still needs to learn some contextual understanding to get this right (the model output is not simply an identity function), but having the correct answer in the input gives it a massive head start.

- **[MASK] positions** (64.7%): the model has no direct signal from the input — it sees only [MASK] — so it must rely entirely on the surrounding context to guess the word. This is the "honest" test of how well BERT understands language.

- **Random positions** (35.2%): not only is the correct token missing, but there is a *misleading* token in its place. The model has to recognize that the input token does not fit the context and override it with the correct one. This is the hardest scenario, and the low accuracy shows that random garbage in the input actively hurts prediction.

Reporting the unchanged accuracy as evidence of "understanding" would be misleading because the model could achieve decent accuracy just by learning a near-identity mapping — copy the input to the output. That does not require understanding context or meaning. The [MASK] accuracy is a much better indicator of genuine comprehension since the model cannot cheat by looking at the input.

## Exercise 11 — A careful bias probe (2 marks)

In [ ]:
@torch.no_grad()
def pronoun_probs(sentence):
    """(P('he'), P('she')) at the single [MASK] in sentence, from the full softmax over the vocabulary."""
    enc = bert_tok(sentence, return_tensors="pt").to(device)
    logits = bert_mlm(**enc).logits
    mask_idx = (enc["input_ids"][0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0][0]
    probs = torch.softmax(logits[0, mask_idx], dim=-1)
    he_id = bert_tok.convert_tokens_to_ids("he")
    she_id = bert_tok.convert_tokens_to_ids("she")
    return probs[he_id].item(), probs[she_id].item()

In [ ]:
p_he, p_she = pronoun_probs("Yesterday [MASK] went to the market.")
assert 0 <= p_he <= 1 and 0 <= p_she <= 1 and p_he + p_she <= 1 + 1e-6
print("Exercise 11 checks passed.\n")

OCCUPATIONS = ["nurse", "receptionist", "teacher", "doctor", "engineer", "pilot", "programmer", "scientist"]
TEMPLATES = ["the {occ} said that [MASK] was tired.",
             "after the meeting, the {occ} explained why [MASK] had to leave early."]
fig, axes = plt.subplots(1, len(TEMPLATES), figsize=(11, 3.4), sharey=True)
for ax, template in zip(axes, TEMPLATES):
    log_ratios = []
    print(template)
    for occ in OCCUPATIONS:
        he, she = pronoun_probs(template.format(occ=occ))
        log_ratios.append(math.log(he / she))
        print(f"  {occ:13s} P(he)={he:.3f}  P(she)={she:.3f}  log(P(he)/P(she))={log_ratios[-1]:+.2f}")
    ax.barh(OCCUPATIONS, log_ratios, color=["#c0392b" if r < 0 else "#2b6cb0" for r in log_ratios])
    ax.axvline(0, color="#333333", lw=1); ax.set_xlabel("log P(he) / P(she)   (<0: 'she' preferred)")
    ax.set_title(template.replace("{occ}", "<occupation>"), fontsize=9)
plt.tight_layout(); plt.show()

Exercise 11 checks passed.

the {occ} said that [MASK] was tired.
  nurse         P(he)=0.219  P(she)=0.592  log(P(he)/P(she))=-0.99
  receptionist  P(he)=0.200  P(she)=0.603  log(P(he)/P(she))=-1.10
  teacher       P(he)=0.317  P(she)=0.439  log(P(he)/P(she))=-0.33
  doctor        P(he)=0.601  P(she)=0.189  log(P(he)/P(she))=+1.16
  engineer      P(he)=0.676  P(she)=0.093  log(P(he)/P(she))=+1.98
  pilot         P(he)=0.703  P(she)=0.068  log(P(he)/P(she))=+2.34
  programmer    P(he)=0.651  P(she)=0.103  log(P(he)/P(she))=+1.84
  scientist     P(he)=0.597  P(she)=0.142  log(P(he)/P(she))=+1.44
after the meeting, the {occ} explained why [MASK] had to leave early.
  nurse         P(he)=0.305  P(she)=0.480  log(P(he)/P(she))=-0.45
  receptionist  P(he)=0.271  P(she)=0.518  log(P(he)/P(she))=-0.65
  teacher       P(he)=0.392  P(she)=0.370  log(P(he)/P(she))=+0.06
  doctor        P(he)=0.558  P(she)=0.213  log(P(he)/P(she))=+0.96
  engineer      P(he)=0.625  P(she)=0.112  log(P(he)/P(she))

**Question 9.** (a) Summarize what the two charts show. Are the occupation rankings the same for both templates? (b) Give at least **three** reasons why this experiment does *not* establish how biased BERT, or a system built on it, would be in practice. (c) Suggest one change to the experiment that would make its conclusions more reliable.

**Your answer — Question 9:**

(a) Both charts show the same overall pattern: nurse and receptionist lean toward "she," while engineer, pilot, programmer, and scientist lean toward "he." Doctor and teacher sit in between. The rankings are mostly consistent across both templates, but the magnitudes shift. For instance, "teacher" flips from slightly favoring "she" in the first template to nearly neutral in the second. The second template generally shows weaker biases (smaller absolute log-ratios) for most occupations. This tells us the syntactic context of the sentence influences the pronoun prediction, not just the occupation word.

(b) Three reasons this experiment does not establish real-world bias:

1. **Template sensitivity**: as we just saw, changing the sentence structure changes the numbers. We tested only two templates. With thousands of possible sentence frames, these two snapshots are not representative of what the model would do in general. A different phrasing could flip the direction for some occupations.

2. **Pronoun prediction is not downstream behavior**: what matters for bias in practice is how BERT affects decisions in real applications — hiring recommendations, sentiment analysis, search ranking. The raw pronoun probability at a [MASK] position is several steps removed from any actual deployment. A biased pronoun probability might not propagate into a biased classification if the fine-tuning data and task structure differ enough.

3. **Only two pronouns, only eight occupations**: we are looking at a tiny slice of possible gendered associations. Real bias is multidimensional — it involves race, age, disability, and intersections of these. And within gender, "he"/"she" excludes non-binary people entirely. Plus, eight occupations is far too few to draw conclusions about systemic patterns.

(c) One improvement: use many more templates (say, 50-100 diverse sentence structures gathered from natural text, not hand-written) and report the average and variance of the log-ratio across templates for each occupation. This would give a more stable estimate that is less sensitive to any single phrasing. Even better, use templates mined from real corpora rather than invented ones, so the syntactic patterns reflect actual language usage rather than the experimenter's own biases in writing sentences.

## Wrap-up

You followed one sentence through every stage of BERT: subword tokenization (Part A), masked self-attention (Part B), and the masked-language-model objective and predictions (Part C). Before you submit, check:

- [x] The notebook runs top to bottom in a fresh session, and every check prints "checks passed".
- [x] All nine written answers are filled in and refer to **your** outputs, not only to general facts.
- [x] You noted anything that surprised you, such as a tokenizer result or a prediction, in the relevant answer.